pydantic

richest feature set 

In [4]:
import os
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model=init_chat_model("google_genai:gemini-2.5-flash-lite")
response = model.invoke("Where is krishnanagar?")
print(response.content)

Krishnanagar is a city located in the **Nadia district of West Bengal, India.**


In [5]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The release year of the movie")
    rating: float = Field(..., description="The rating of the movie")
    

In [7]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x000001EF0F7F2CF0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type':

In [8]:
model.invoke("What is the movie 'Inception' about?")

AIMessage(content='"Inception" is a mind-bending science fiction action film directed by Christopher Nolan. At its core, the movie is about a team of skilled thieves who specialize in **extracting valuable information by entering people\'s subconscious through shared dreaming.**\n\nHowever, the premise of the movie takes a significant turn when they are offered a job that is the inverse of their usual work: **inception**. Instead of stealing an idea, they must **plant an idea** into the mind of a target. This is considered a much more dangerous and complex undertaking, as it requires not just extracting, but subtly influencing and embedding a new thought deep within the subconscious.\n\nHere\'s a breakdown of the key elements:\n\n*   **The Team:** Led by Dom Cobb (Leonardo DiCaprio), a master extractor haunted by his past and unable to return home due to a past event. He assembles a specialized team with unique skills for navigating the dreamscape:\n    *   **Arthur (Joseph Gordon-Levi

In [9]:
#output
response = model_with_structure.invoke("What is the movie 'Inception' about?")
print(response)

Movie(title='Inception', director='Christopher Nolan', release_year=2010, rating=8.8)

### message output parsed struc

In [11]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    ### Movie Schema ###
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The release year of the movie")
    rating: float = Field(..., description="The rating of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("What is the movie 'Inception' about?")
print(response)

{'raw': AIMessage(content='{\n  "title": "Inception",\n  "director": "Christopher Nolan",\n  "release_year": 2010,\n  "rating": 8.8\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e84e1-7ad2-7812-a66f-711a84873c81-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 43, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}}), 'parsed': Movie(title='Inception', director='Christopher Nolan', release_year=2010, rating=8.8), 'parsing_error': None}


### nested struc

In [12]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    age: int = Field(..., description="The age of the actor")
    movies: list[str] = Field(..., description="List of movies the actor has acted in")
class MovieDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The release year of the movie")
    rating: float = Field(..., description="The rating of the movie")
    actors: list[Actor] = Field(..., description="List of actors in the movie")
    budget: float = Field(..., description="The budget of the movie")
model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("What is the movie 'Inception' about?")
print(response)

{'raw': AIMessage(content='{\n  "title": "Inception",\n  "director": "Christopher Nolan",\n  "release_year": 2010,\n  "rating": 8.8,\n  "actors": [\n    {\n      "name": "Leonardo DiCaprio",\n      "age": 49,\n      "movies": ["Titanic", "The Wolf of Wall Street", "Inception", "Shutter Island"]\n    },\n    {\n      "name": "Joseph Gordon-Levitt",\n      "age": 43,\n      "movies": ["Inception", "The Dark Knight Rises", "500 Days of Summer"]\n    },\n    {\n      "name": "Elliot Page",\n      "age": 37,\n      "movies": ["Juno", "Inception", "X-Men: Days of Future Past"]\n    }\n  ],\n  "budget": 160000000\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e84e3-4418-7381-a8d0-d91d1e97c3f2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 215, 'total_tokens': 226, 'input_token_details': {'cache_read': 0}}), 'pars

Typedict

### runtime validation not required

In [18]:
from typing_extensions import TypedDict, Annotated
class Movie(TypedDict):
    title: Annotated[str, "The title of the movie"]
    director: Annotated[str, "The director of the movie"]
    release_year: Annotated[int, "The release year of the movie"]
    rating: Annotated[float, "The rating of the movie"]

model_with_typedict = model.with_structured_output(Movie)
response = model_with_typedict.invoke("What is the movie 'Avengers' about?")
print(response)

{'title': 'The Avengers', 'director': 'Joss Whedon', 'release_year': 2012, 'rating': 8.0}


In [20]:
class Actor(TypedDict):
    name: Annotated[str, "The name of the actor"]
    age: Annotated[int, "The age of the actor"]
    movies: Annotated[list[str], "List of movies the actor has acted in"]
class MovieDetails(TypedDict):
    title: Annotated[str, "The title of the movie"]
    director: Annotated[str, "The director of the movie"]
    release_year: Annotated[int, "The release year of the movie"]
    rating: Annotated[float, "The rating of the movie"]
    actors: Annotated[list[Actor], "List of actors in the movie"]
    budget: Annotated[float, "The budget of the movie"]
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("What is the movie 'Inception' about?")
print(response)

{'title': 'Inception'}


In [22]:
model.profile

{'name': 'Gemini 2.5 Flash-Lite',
 'release_date': '2025-06-17',
 'last_updated': '2025-06-17',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

### dataclass

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [30]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

class ContactInfo(BaseModel):
    name: str = Field(..., description="The name of the person")
    email: str = Field(..., description="The email address of the person")
    phone: str = Field(..., description="The phone number of the person")

agent_executor = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo
)
result = agent_executor.invoke({
    "messages": [
        {"role": "system", "content": "You are a helpful assistant that provides contact information."},
        {"role": "user", "content": "What is the contact information of John Doe?"}
    ]
})
print(result)

{'messages': [SystemMessage(content='You are a helpful assistant that provides contact information.', additional_kwargs={}, response_metadata={}, id='28dc0926-7c19-473c-93d0-d1bf7262a0bc'), HumanMessage(content='What is the contact information of John Doe?', additional_kwargs={}, response_metadata={}, id='fe8f3b3b-1b5c-406d-97dd-0a0395d193c2'), AIMessage(content='{\n  "name": "John Doe",\n  "email": "john.doe@example.com",\n  "phone": "123-456-7890"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e84f4-cc8a-7d31-abad-f7effe35d34f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 45, 'total_tokens': 65, 'input_token_details': {'cache_read': 0}})], 'structured_response': ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')}


In [1]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

agent=create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo
)
result = agent.invoke({
    "messages": [
        {"role": "system", "content": "You are a helpful assistant that provides contact information."},
        {"role": "user", "content": "What is the contact information of John Doe?"}
    ]
})
print(result)

{'messages': [SystemMessage(content='You are a helpful assistant that provides contact information.', additional_kwargs={}, response_metadata={}, id='baae5196-f6f5-4bd4-9aac-f2ed2c5d4678'), HumanMessage(content='What is the contact information of John Doe?', additional_kwargs={}, response_metadata={}, id='a5755cd8-b40c-46e8-af0c-57984bcfafd1'), AIMessage(content='{\n  "name": "John Doe",\n  "email": "john.doe@example.com",\n  "phone": "123-456-7890"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e84f7-c5bf-7522-8753-9d138abd074e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 45, 'total_tokens': 65, 'input_token_details': {'cache_read': 0}})], 'structured_response': ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')}
